# Data Loading and Preparation

## Basic Imports

In [ ]:
# Data handling
import pandas as pd

# Pathing
from pathlib import Path

# Modelling
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import joblib # For persistence

## Load Processed Feature Dataset for Modelling

In [ ]:
df_model_path = Path("../data/processed/stage2_features.csv")
df_model = pd.read_csv(df_model_path)
print("Loaded dataset shape:", df_model.shape)
display(df_model.head())

### Cast to Category for Native Handling

In [ ]:
categorical_cols = ["state", "service", "proto", "dsport"]
for col in categorical_cols:
    df_model[col] = df_model[col].astype("category")

## Train/Test Split

### Create Split

In [ ]:
X = df_model.drop("Label", axis=1)
y = df_model["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y, # Preserve class balance
    random_state=42 # Reproducibility
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

### Save Split

In [ ]:
split_path = Path("../data/splits")
split_path.mkdir(parents=True, exist_ok=True)
X_train.to_csv(split_path / "X_train.csv", index=False)
X_test.to_csv(split_path / "X_test.csv", index=False)
y_train.to_csv(split_path / "y_train.csv", index=False)
y_test.to_csv(split_path / "y_test.csv", index=False)
print(f"Train/test split saved to {split_path}.")

# Model Fit

## XGBoost Classifier Model

### Initialise Model

In [ ]:
model = XGBClassifier(
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    max_depth=6,
    learning_rate=0.03,
    n_estimators=900,
    subsample=0.9,
    colsample_bytree=0.6,
    min_child_weight=2,
    gamma=0.2,
    enable_categorical=True
)

Regularisation and subsampling hyperparameters were adjusted to improve generalisation. This allows the semantically relevant `sttl` feature to be retained while avoiding over-reliance.

#### Cross-Validation Evaluation

Assess generalisation and tune hyperparameters.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # Handles imbalanced classes
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_weighted")
print("CV scores:", scores)
print(f"CV Weighted F1-score: {scores.mean():.4f}")

### Fit Model

In [ ]:
print("Fitting xgboost classifer...")
model.fit(X_train, y_train)
print("Model fitting complete")

# Model Persistence

In [ ]:
model_path = Path("../models/xgboost_classifier_model.pkl")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}.")